In [31]:
import numpy as np
import numba as nb
import json
import os
import subprocess


The nematic order, $Q_{ij}$, is given by

$$Q_{ij} = S(n_i n_j - \frac{1}{2}).$$

$n_i$ is a unit vector. Thus, $Q_{ij}$ is traceless and symmetric, giving it 2 degrees of freedom,

$$Q_{ij} = \begin{pmatrix}
Q_{xx} & Q_{xy}  \\
Q_{xy} & -Q_{xx} 
\end{pmatrix}.$$

$$Q_{xx} = S(n_x^2 - 1/2),$$
$$Q_{xy} = S(n_x n_y).$$

The time evolution of $Q_{ij}$ is given by gradient descent:

$$\partial_t Q_{ij} = dt (- \frac{\delta F_{\mathrm{LdG}}}{\partial Q_{ij}} ) = K (\partial_k^2 Q_{ij} + \ell_c^{-2} Q_{ij} (1 - 2 \mathrm{Tr}[Q^2])).$$

$$\mathrm{Tr}[Q^2] = \mathrm{Tr}\left[ \begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\right] = \mathrm{Tr}\left[\begin{pmatrix}Q_{xx}^2 + Q_{xy}^2 & 0 \\ 0 & Q_{xx}^2 + Q_{xy}^2 \end{pmatrix}\right] = 2(Q_{xx}^2 + Q_{xy}^2).$$

In [32]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_dQdt(dQdt, Q, A, K, neighbors, points):

    for p in nb.prange(points):

        # coordinates of nearest neighbors and diagonal points
        xup, xdn, yup, ydn = neighbors[p]
        xup_yup, xup_ydn = neighbors[xup][2:]
        xdn_yup, xdn_ydn = neighbors[xdn][2:]

        # \nabla^2 Q
        lap_Q = (( Q[xup] + Q[xdn] + Q[yup] + Q[ydn]) * 0.666666667
                +( Q[xup_yup] + Q[xup_ydn] + Q[xdn_yup] + Q[xdn_ydn]) * 0.166666667
                -  Q[p] * 3.333333333)
        
        # Tr[Q^2]
        TrQ2 = 2 * (Q[p, 0] * Q[p, 0] + Q[p, 1] * Q[p, 1])
        
        # dQ/dt = -dF/dQ * dt 
        # -dF/dQ = K (\nabla^2 Q + \ell^c^{-2} Q (1 - 2 Tr[Q^2]))
        dQdt[p] = K * (lap_Q + A * Q[p] * (1 - 2 * TrQ2))

In [33]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def randomize(Q, points):
    
    for p in nb.prange(points):

        # Completely Random Director Everywhere
        phi = np.random.uniform(0, 2 * np.pi)

        Q[p, 0] = 0.5 * np.cos(2 * phi)
        Q[p, 1] = 0.5 * np.sin(2 * phi)

In [34]:
def make_boundary(boundary, Lx, Ly):
    
    neighbors = []
    if boundary == "periodic":
        for p in range(Lx * Ly):
            x = p // Ly
            y = p % Ly

            xup = ((x + 1) % Lx) * Ly + y
            xdn = ((x - 1) % Lx) * Ly + y
            yup = x * Ly + (y + 1) % Ly
            ydn = x * Ly + (y - 1) % Ly
            neighbors.append((xup, xdn, yup, ydn))

    else:
        raise ValueError("Unknown boundary condition")
    return np.array(neighbors)

In [35]:
Lx = Ly      = 2**7 - 1        # system size
T            = int(5e4)        # max time steps
K            = 1/2**3          # elastic modulus
A            = 1/(1)**2        # de Gennes constant
boundary     = "periodic"      # boundary condition
runname      = f"medium"       # output directory

In [36]:
if __name__ == "__main__":

    nb.set_num_threads(os.cpu_count() - 1)

    subprocess.run(f"mkdir -p {runname}", shell=True)
    subprocess.run(f"mkdir -p {runname}/data/", shell=True)
    json.dump(
        {
            "System Size":Lx,
            "Elasticity":K,
            "Boundry Conditions":boundary,
            "DeGennes Constant": A
        }, open(f"{runname}/{runname}.json", 'w'), indent = 4
    )

    neighbors = make_boundary(boundary, Lx, Ly)
    points    = len(neighbors)

    Q         = np.zeros((points, 2))  # nematic order parameter
    dQdt      = np.zeros((points, 2))  # time derivative of Q

    randomize(Q, points)

    for t in range(T):
        
        get_dQdt(dQdt, Q, A, K, neighbors, points)
        Q += dQdt

        if t % 100 == 0: np.savez(f"{runname}/data/{t:10d}.npz", Q=Q)

    subprocess.run(f"python3 plot_2D_Fmin.py {runname}", shell=True)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex